# Citiz — 5. Modèle : où Citiz s'installerait-il ? (Gold)
Runtime Python. À exécuter après `acces_notebook` et avant `optimisation_notebook`.
**Cible** : pour chaque carreau habité de métropole, présence d'une station Citiz à 400 m (classification).
Le modèle apprend **où Citiz s'est installé** (≈ 1 600 stations), pas une demande mesurée : l'historique
d'usage Citiz est inexploitable. Aucune variable n'est calculée à partir des stations Citiz (pas de fuite).
**Validation** : par région (on retire une région entière et on la prédit) ; test à l'aveugle sur Grenoble
(Isère absente de l'apprentissage) ; comparaison à une baseline « densité seule » ; contributions SHAP.
**Sortie** : `proba_citiz` par carreau, lue par l'optimisation comme usage potentiel des stations Citiz.

In [ ]:
# deltalake 1.x comme les autres notebooks (la version préinstallée ne lit pas les tables écrites par Bronze) ;
# pas de scikit-learn : le remplacer par-dessus celui de Fabric casse les imports.
%pip install -q "deltalake>=1,<2" "lightgbm==4.6.0"

In [ ]:
SILVER = 'silver'
GOLD = 'gold'
SCHEMA = 'dbo'
LOCAL_ROOT = None
RAYONS_M = [400, 1000]
DEPT_AVEUGLE = '38'          # Isère : Grenoble, test à l'aveugle
# Modèle volontairement simple et monotone : plus de population ou d'offre ne baisse jamais le potentiel.
# Les variables communales (population, densité, croissance de la commune) sont écartées : elles faisaient
# apprendre « quelles communes ont Citiz » et se transposaient mal d'une région à l'autre.
PARAMS = {'objective': 'binary', 'learning_rate': 0.05, 'num_leaves': 15, 'min_data_in_leaf': 1000,
          'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1, 'verbose': -1, 'seed': 42}
NB_ARBRES = 200
TAUX_NEGATIFS = 0.1          # part des carreaux sans Citiz gardés pour l'apprentissage (2 cœurs dans Fabric)
ECHANTILLON_SHAP = 50000

In [ ]:
import os, json, uuid, datetime as dt
import numpy as np
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from scipy.spatial import cKDTree
import lightgbm as lgb
RUN_ID = str(uuid.uuid4())
STATUS = []

def nu():
    import notebookutils
    return notebookutils

def root(lakehouse):
    if LOCAL_ROOT:
        return f"{LOCAL_ROOT}/{lakehouse}", None
    info = nu().lakehouse.get(lakehouse)
    if not isinstance(info, dict):
        info = json.loads(json.dumps(info, default=lambda x: x.__dict__))
    props = info.get("properties", {})
    path = props.get("abfsPath")
    if not path:
        raise RuntimeError(f"abfsPath absent pour {lakehouse}; contrôler notebookutils.lakehouse.get")
    return path.rstrip("/"), {"bearer_token": nu().credentials.getToken("storage"), "use_fabric_endpoint":"true"}

def table_path(lakehouse, name):
    p,o = root(lakehouse)
    return f"{p}/Tables/{SCHEMA + '/' if SCHEMA else ''}{name}",o

def read_table(lakehouse, name, required=False, run_id=None, columns=None):
    p,o = table_path(lakehouse,name)
    try:
        dt_=DeltaTable(p,storage_options=o)
        if columns: columns=[c for c in columns+['_run_id'] if c in {f.name for f in dt_.schema().fields}]
        t=dt_.to_pandas(columns=columns)
    except Exception as e:
        if required: raise RuntimeError(f"Table obligatoire illisible: {lakehouse}.{name}") from e
        print(f"Absente/illisible: {lakehouse}.{name}: {str(e)[:120]}")
        return pd.DataFrame()
    if run_id is not None:
        if '_run_id' not in t: return pd.DataFrame()
        t=t[t['_run_id']==run_id].copy()
    return t.drop(columns=['_run_id'],errors='ignore')

def write_table(lakehouse,name,df):
    df=df.reset_index(drop=True).copy()
    df['_run_id']=RUN_ID
    # Types explicites même pour colonnes entièrement nulles / tables vides.
    a=pa.Table.from_pandas(df,preserve_index=False)
    a=pa.Table.from_arrays([a[c].cast(pa.string()) if pa.types.is_null(a[c].type) else a[c]
                            for c in a.column_names],names=a.column_names)
    p,o=table_path(lakehouse,name)
    if LOCAL_ROOT: os.makedirs(os.path.dirname(p),exist_ok=True)
    write_deltalake(p,a,mode='overwrite',schema_mode='overwrite',storage_options=o)
    STATUS.append({'table':name,'lignes':len(df),'run_id':RUN_ID})
    print(f"{lakehouse}.{name}: {len(df):,} lignes")

def xyz(lon,lat):
    a,b=np.radians(np.asarray(lon,dtype=float)),np.radians(np.asarray(lat,dtype=float))
    return np.column_stack([np.cos(b)*np.cos(a),np.cos(b)*np.sin(a),np.sin(b)])

def chord(metres):
    return 2*np.sin(metres/(2*6371008.8))

def roc_auc_score(y, s):
    # Mann-Whitney : probabilité qu'un carreau Citiz soit classé avant un carreau sans Citiz (ex aequo à moitié).
    y=np.asarray(y); r=pd.Series(np.asarray(s,dtype=float)).rank().to_numpy(); n1=y.sum(); n0=len(y)-n1
    return float((r[y==1].sum()-n1*(n1+1)/2)/(n1*n0))

def average_precision_score(y, s):
    # Précision moyenne aux rangs des positifs, par seuil distinct (même définition que scikit-learn).
    y=np.asarray(y); s=np.asarray(s,dtype=float); o=np.argsort(-s,kind='mergesort'); y,s=y[o],s[o]
    last=np.r_[np.flatnonzero(np.diff(s)),len(s)-1]; tp=np.cumsum(y)[last]; prec=tp/(last+1)
    return float(np.sum(np.diff(np.r_[0,tp])*prec)/y.sum())

## 1 — Données : carreaux de métropole et offre hors Citiz

In [ ]:
run=read_table(SILVER,'pipeline_run',True)
RUN_ID=run.iloc[0]['run_id']
cells=read_table(GOLD,'gold_acces_carreau',True,RUN_ID,columns=['carreau_id','commune_code','lat','lon','pop_calee',
                 'affichable','zone_focus','couvert_citiz','niveau_vie_moyen'])
if cells.empty: raise ValueError('gold_acces_carreau absent pour ce run : lancer acces_notebook')
offer=read_table(GOLD,'gold_offre_desservie',True,RUN_ID,columns=['famille','operateur','lat','lon'])
com=read_table(SILVER,'dim_commune',True,RUN_ID,columns=['commune_code','code_dept','code_region'])
cars=read_table(SILVER,'fact_motorisation',True,RUN_ID)
cells=cells[~cells.commune_code.str.startswith('97')].reset_index(drop=True)   # métropole : Citiz n'est pas en DROM
cells=cells.merge(com,on='commune_code',how='left')
print(f"{len(cells):,} carreaux, {cells.couvert_citiz.sum():,} avec Citiz à 400 m")

## 2 — Variables (aucune ne dépend des stations Citiz)
Population autour du carreau (400 m, 1 km), offre de TC, de vélos et de free-floating à 400 m et 1 km,
distance à l'arrêt le plus proche, part de ménages sans voiture de la commune, niveau de vie moyen du carreau
(Filosofi ; sans contrainte de sens). Les tranches d'âge ont été testées : elles n'améliorent pas la validation.

In [ ]:
X=pd.DataFrame(index=cells.index)
X['pop_carreau']=cells.pop_calee
cell_tree=cKDTree(xyz(cells.lon,cells.lat))
q=xyz(cells.lon,cells.lat)
for r in RAYONS_M:
    tot=np.zeros(len(cells))
    for s in range(0,len(cells),200000):
        pairs=cKDTree(q[s:s+200000]).sparse_distance_matrix(cell_tree,chord(r),output_type='ndarray')
        np.add.at(tot,pairs['i']+s,cells.pop_calee.to_numpy()[pairs['j']])
    X[f'pop_{r}m']=tot
for fam in ['tc','velo','free_floating']:
    pts=offer[offer.famille.eq(fam)]
    tree=cKDTree(xyz(pts.lon,pts.lat))
    for r in RAYONS_M:
        X[f'{fam}_{r}m']=tree.query_ball_point(q,chord(r),return_length=True)
d,_=cKDTree(xyz(offer[offer.famille.eq('tc')].lon,offer[offer.famille.eq('tc')].lat)).query(q,k=1)
X['distance_tc_m']=2*6371008.8*np.arcsin(np.clip(d/2,0,1))
sv=cars.sort_values('annee').drop_duplicates('commune_code',keep='last').set_index('commune_code').part_sans_voiture
X['part_sans_voiture']=cells.commune_code.map(sv)
if cells.niveau_vie_moyen.notna().any(): X['niveau_vie_moyen']=cells.niveau_vie_moyen
y=cells.couvert_citiz.astype(int).to_numpy()
FEATURES=list(X.columns)
PARAMS['monotone_constraints']=[-1 if c=='distance_tc_m' else (0 if c=='niveau_vie_moyen' else 1) for c in FEATURES]
print(FEATURES)

## 3 — Validation
- **Par région** : chaque région métropolitaine est prédite par un modèle qui ne l'a jamais vue.
- **Grenoble à l'aveugle** : modèle entraîné sans l'Isère ; on mesure s'il retrouve les carreaux Citiz.
- **Baseline « densité seule »** : classer par population à 1 km. Le modèle doit faire mieux.
Métriques : AUC (classement) et précision moyenne (PR-AUC, adaptée aux classes rares).
Résultat du 20/10/2026 (run Fabric) : la densité seule reste presque aussi bonne région par région (PR-AUC médiane
0,42 contre 0,37) et à Grenoble (0,749 contre 0,746 pour le modèle). Le modèle classe mieux le haut de liste à
Grenoble (67,5 % de carreaux Citiz parmi les plus probables, contre 63,7 %) et donne une probabilité utilisable
par l'optimisation. Citiz s'installe d'abord là où il y a du monde à pied ; le niveau de vie affine le haut de liste.

In [ ]:
tirage=np.random.default_rng(42).random(len(y))

def fit(mask):
    # Tous les carreaux Citiz + 10 % des autres, pondérés par 10 : même classement, probabilités calibrées.
    keep=mask&((y==1)|(tirage<TAUX_NEGATIFS))
    w=np.where(y[keep]==1,1.0,1/TAUX_NEGATIFS)
    return lgb.train(PARAMS,lgb.Dataset(X[keep],y[keep],weight=w),NB_ARBRES)

def scores(name,mask,p):
    yt=y[mask]
    if yt.sum()==0 or yt.sum()==len(yt): return None
    return {'evaluation':name,'carreaux':int(mask.sum()),'carreaux_citiz':int(yt.sum()),
            'auc_modele':roc_auc_score(yt,p),'auc_densite_seule':roc_auc_score(yt,X.pop_1000m[mask]),
            'pr_auc_modele':average_precision_score(yt,p),'pr_auc_densite_seule':average_precision_score(yt,X.pop_1000m[mask]),
            'taux_citiz':float(yt.mean())}

rows=[]
region=cells.code_region.fillna('?').to_numpy()
# proba_oof : probabilité prédite par un modèle qui n'a pas vu la région du carreau (ni donc son réseau Citiz).
# C'est elle que l'optimisation utilise : un modèle qui a vu les stations nantaises dévaloriserait les zones à conquérir.
proba_oof=np.full(len(y),np.nan)
for reg in sorted(set(region)):
    test=region==reg
    if y[test].sum()<20: continue   # région sans réseau Citiz significatif
    proba_oof[test]=fit(~test).predict(X[test])
    rows.append(scores(f'region {reg}',test,proba_oof[test]))
blind=cells.code_dept.eq(DEPT_AVEUGLE).to_numpy()
m_blind=fit(~blind)
gre=blind&cells.zone_focus.eq('Grenoble-Alpes-Métropole').to_numpy()
p_gre=m_blind.predict(X[gre])
rows.append(scores('Grenoble a l aveugle',gre,p_gre))
# Lecture concrète : parmi les carreaux jugés les plus probables (autant que de carreaux Citiz réels), combien en ont une ?
k=int(y[gre].sum()); top=np.argsort(-p_gre)[:k]
rows[-1]['part_top_k_avec_citiz']=float(y[gre][top].mean())
rows[-1]['part_top_k_densite_seule']=float(y[gre][np.argsort(-X.pop_1000m[gre].to_numpy())[:k]].mean())
valid=pd.DataFrame([r for r in rows if r])
print(valid.round(3).to_string(index=False))

## 4 — Modèle final, probabilités et contributions SHAP
Entraîné sur toute la métropole. `proba_citiz` sert d'usage potentiel dans l'optimisation ;
`classe_potentiel` (quintiles parmi les carreaux de la même zone) colore la carte de Nantes.

In [ ]:
model=fit(np.ones(len(y),bool))
cells['proba_citiz']=model.predict(X)
cells['proba_oof']=proba_oof   # vide dans les régions sans réseau Citiz significatif : l'optimisation prend alors proba_citiz
rng=np.random.default_rng(42)
idx=rng.choice(len(X),min(ECHANTILLON_SHAP,len(X)),replace=False)
contrib=model.predict(X.iloc[idx],pred_contrib=True)[:,:-1]   # SHAP natif de LightGBM, sans la constante
imp=pd.DataFrame({'variable':FEATURES,'shap_moyen_abs':np.abs(contrib).mean(axis=0),
                  'gain':model.feature_importance('gain')}).sort_values('shap_moyen_abs',ascending=False)
imp['part_shap']=imp.shap_moyen_abs/imp.shap_moyen_abs.sum()
print(imp.round(4).to_string(index=False))
z=cells.zone_focus.fillna('France')
cells['rang_pct_zone']=cells.groupby(z).proba_citiz.rank(pct=True)
cells['classe_potentiel']=pd.cut(cells.rang_pct_zone,[0,.2,.4,.6,.8,1],
                                 labels=['très faible','faible','moyen','fort','très fort'],include_lowest=True).astype(str)
assert cells.proba_citiz.between(0,1).all() and len(valid)>0
out=pd.concat([cells[['carreau_id','commune_code','lat','lon','pop_calee','affichable','zone_focus','couvert_citiz',
                      'proba_citiz','proba_oof','rang_pct_zone','classe_potentiel']],X.drop(columns=['pop_carreau','niveau_vie_moyen'],errors='ignore')],axis=1)
write_table(GOLD,'gold_modele_proba_carreau',out)
write_table(GOLD,'gold_modele_validation',valid)
write_table(GOLD,'gold_modele_importance',imp)
print('Modèle prêt : proba_citiz = ressemblance aux lieux où Citiz est installé, pas une demande mesurée.')